In [ ]:
!pip install ultralytics albumentations -q

In [ ]:
import os, shutil, yaml, random
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import albumentations as A
from ultralytics import YOLO

In [ ]:
TRAIN_CSV   = "/kaggle/input/competitions/object-detetction-week-10-dlp/train.csv"
TRAIN_IMGS  = "/kaggle/input/competitions/object-detetction-week-10-dlp/train"
TEST_IMGS   = "/kaggle/input/competitions/object-detetction-week-10-dlp/test"
WORK_DIR    = "/kaggle/working"
LABELS_DIR  = f"{WORK_DIR}/labels"
YAML_PATH   = f"{WORK_DIR}/data.yaml"

In [ ]:
# ============================================================
#  Dataset Curation
#  - Normalize coords
#  - Train/Val split (stratified by class)
#  - Write YOLO labels (vectorized)
# ============================================================
df = pd.read_csv(TRAIN_CSV)

# Class map
CLASSES = [
    "RiceLeafRoller", "RiceLeafCaterpillar", "PaddyStemMaggot",
    "AsiaticRiceBorer", "YellowRiceBorer", "RiceGallMidge",
    "RiceStemfly", "BrownPlantHopper", "WhiteBackedPlantHopper",
    "SmallBrownPlantHopper", "RiceWaterWeevil", "RiceLeafhopper",
    "GrainSpreaderThrips", "RiceShellPest", "Grub",
    "MoleCricket", "Wireworm", "WhiteMarginedMoth",
    "BlackCutworm", "LargeCutworm", "YellowCutworm",
    "RedSpider", "CornBorer", "Armyworm"
]
classes     = CLASSES
class_to_id = {c: i for i, c in enumerate(classes)}
print(f"Classes ({len(classes)}): {classes}")

# ── Vectorized normalization ──────────────────────────────────
df['class_id'] = df['class_name'].map(class_to_id)
df['x_center'] = ((df['x_min'] + df['x_max']) / 2) / df['width']
df['y_center'] = ((df['y_min'] + df['y_max']) / 2) / df['height']
df['w_norm']   = (df['x_max'] - df['x_min']) / df['width']
df['h_norm']   = (df['y_max'] - df['y_min']) / df['height']

# ── Clip & drop bad boxes ─────────────────────────────────────
norm_cols = ['x_center', 'y_center', 'w_norm', 'h_norm']
df[norm_cols] = df[norm_cols].clip(0, 1)
df = df[(df['w_norm'] > 0) & (df['h_norm'] > 0)].copy()

In [ ]:
# ── Stratified Train / Val split (80/20 by image) ────────────
#  Use the dominant class per image as the stratification key
dominant_class = (
    df.groupby('image_id')['class_name']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
unique_imgs = dominant_class['image_id'].values
strat_labels = dominant_class['class_name'].values

train_imgs, val_imgs = train_test_split(
    unique_imgs,
    test_size=0.2,
    stratify=strat_labels,
    random_state=42
)
print(f"Train images: {len(train_imgs)} | Val images: {len(val_imgs)}")

In [ ]:
# ── Build YOLO line strings vectorized ────────────────────────
df['yolo_line'] = (
    df['class_id'].astype(str)        + " " +
    df['x_center'].round(6).astype(str) + " " +
    df['y_center'].round(6).astype(str) + " " +
    df['w_norm'].round(6).astype(str)   + " " +
    df['h_norm'].round(6).astype(str)
)

# ── Write labels to train/ or val/ ───────────────────────────
train_set = set(train_imgs)
for split in ['train', 'val']:
    os.makedirs(f"{LABELS_DIR}/{split}", exist_ok=True)

def write_label(group):
    img_id   = group.name
    split    = 'train' if img_id in train_set else 'val'
    stem     = Path(img_id).stem
    out_path = f"{LABELS_DIR}/{split}/{stem}.txt"
    with open(out_path, "w") as f:
        f.write("\n".join(group['yolo_line'].values))

df.groupby('image_id').apply(write_label)
print(f"  Labels written to: {LABELS_DIR}/train & /val")

In [ ]:
for split, img_list in [("train", train_imgs), ("val", val_imgs)]:
    split_dir = Path(f"{WORK_DIR}/images/{split}")
    split_dir.mkdir(parents=True, exist_ok=True)
    for img_name in img_list:
        src = Path(f"{TRAIN_IMGS}/{img_name}")
        dst = split_dir / img_name
        if not dst.exists():
            os.symlink(src, dst)
print(" Image symlinks created")

In [ ]:
# ── Save classes.txt ─────────────────────────────────────────
with open(f"{WORK_DIR}/classes.txt", "w") as f:
    f.write("\n".join(classes))

# ── Class distribution audit ─────────────────────────
print("\n Class Distribution:")
cc = df['class_name'].value_counts()
for c, n in cc.items():
    print(f"  {c:<30} {n}")
imbalance = cc.max() / cc.min()
print(f"\n  Imbalance ratio: {imbalance:.1f}x")
if imbalance > 10:
    print("  → Consider cls_weight in training args")

In [ ]:
# ============================================================
#  Custom Albumentations Augmentation
#  Applied to minority classes (if imbalance > 5x)
#  NOTE: YOLO's built-in mosaic/HSV/flip handles the rest
# ============================================================

#  The augmentation pipeline (applied manually only if needed)
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.CoarseDropout(num_holes_range=(2, 4), hole_height_range=(10, 20), hole_width_range=(10, 20), p=0.2),
    A.Rotate(limit=10, p=0.3),
], bbox_params=A.BboxParams(
    format='yolo',           # albumentations accepts yolo format directly
    label_fields=['class_labels'],
    min_visibility=0.3
))

print("  Albumentations pipeline ready (for offline augmentation if needed)")
print("    YOLO built-in: mosaic, HSV shift, fliplr, scale — all enabled by default")

In [ ]:
# Custom hyp dict (passed to model.train)
# LR follows cosine decay automatically in Ultralytics
hyp = {
    # ── Learning rate ──────────────────────────────────────
    "lr0"       : 0.01,       # initial LR (Stage 1: frozen backbone)
    "lrf"       : 0.01,       # final LR = lr0 * lrf (cosine decay target)
    "momentum"  : 0.937,
    "weight_decay": 0.0005,

    # ── Augmentation  ─────────────────────────────
    "hsv_h"     : 0.015,      # hue shift
    "hsv_s"     : 0.7,        # saturation shift
    "hsv_v"     : 0.4,        # value shift
    "degrees"   : 5.0,        # rotation ±5°
    "translate"  : 0.1,
    "scale"     : 0.5,
    "flipud"    : 0.0,        # insects can be upside down — set 0.3 if needed
    "fliplr"    : 0.5,
    "mosaic"    : 1.0,        # keep mosaic enabled (great for small pests)
    "mixup"     : 0.1,

    # ── Loss weights ───────────────────────────────────────
    "box"       : 7.5,
    "cls"       : 0.5,
    "dfl"       : 1.5,
}


In [ ]:
# Save as hyp.yaml for reference
with open(f"{WORK_DIR}/hyp.yaml", "w") as f:
    yaml.dump(hyp, f)
print(" Hyperparameters saved: hyp.yaml")

In [ ]:
# Symlink val images to a val folder so YOLO can find them
# (Kaggle images are in a flat train/ folder, labels are split)
# We tell YOLO the image root and override label paths via yaml

data = {
    "path"  : WORK_DIR,
    "train" : f"{WORK_DIR}/images/train",
    "val"   : f"{WORK_DIR}/images/val",        # same img folder — YOLO uses label splits
    "nc"    : len(classes),
    "names" : classes,
}

# Override label dirs explicitly
data["train_labels"] = f"{LABELS_DIR}/train"
data["val_labels"]   = f"{LABELS_DIR}/val"

with open(YAML_PATH, "w") as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(f"  data.yaml saved")

In [ ]:
import torch
torch.cuda.empty_cache()
print(" CUDA cache cleared before Stage 1")

In [ ]:
# ============================================================
#  2-STAGE TRAINING
#
#  STAGE 1 — Frozen Backbone (Transfer Learning)
#    • freeze=10  → only detection head trains
#    • Higher LR  → head weights initialized randomly, need fast learning
#    • Fewer epochs (20–30)
#    • Goal: get the head oriented to your domain quickly
#
#  STAGE 2 — Full Fine-Tune
#    • unfreeze all layers
#    • Lower LR   → backbone pretrained weights are sensitive
#    • More epochs (50–100)
#    • Goal: end-to-end refinement for final accuracy
# ============================================================

# ── Load pretrained model  ──────────────────────────
STAGE1_LAST = f"{WORK_DIR}/runs/stage1_frozen/weights/last.pt"
STAGE1_BEST = f"{WORK_DIR}/runs/stage1_frozen/weights/best.pt"

if Path(STAGE1_LAST).exists():
    print(" Stage 1 checkpoint found — resuming...")
    model = YOLO(STAGE1_LAST)
    stage1 = model.train(resume=True)   # resumes from exact epoch
else:
    print("  Stage 1 starting fresh...")
    model = YOLO("yolov8m.pt")
    stage1 = model.train(
        data        = YAML_PATH,
        epochs      = 25,             # short — just orient the head
        imgsz       = 768,
        batch       = 8,             
        workers     = 2,
        device      = [0,1],
        save_period = 10,
    
        # ── Freeze backbone ──────────────────────────
        freeze      = 10,             # freeze first 10 layers (backbone)
    
        # ── LR for head training ─────────────────────
        lr0         = 0.01,
        lrf         = 0.1,            # cosine decay to lr0 * lrf
        momentum    = hyp["momentum"],
        weight_decay= hyp["weight_decay"],
    
        # ── Augmentation params ────────────────────────
        hsv_h       = hyp["hsv_h"],
        hsv_s       = hyp["hsv_s"],
        hsv_v       = hyp["hsv_v"],
        degrees     = hyp["degrees"],
        translate   = hyp["translate"],
        scale       = hyp["scale"],
        fliplr      = hyp["fliplr"],
        mosaic      = hyp["mosaic"],
        mixup       = hyp["mixup"],
    
        # ── Monitoring ─────────────────────────────────
        plots       = True,           # saves confusion matrix, PR curve
        save        = True,
        project     = f"{WORK_DIR}/runs",
        name        = "stage1_frozen",
        exist_ok    = True,
    )

print("\n Stage 1 complete")
stage1_best = f"{WORK_DIR}/runs/stage1_frozen/weights/best.pt"

In [ ]:
torch.cuda.empty_cache()
print(" CUDA cache cleared before Stage 2")

In [ ]:
STAGE2_LAST = f"{WORK_DIR}/runs/stage2_finetune/weights/last.pt"
STAGE2_BEST = f"{WORK_DIR}/runs/stage2_finetune/weights/best.pt"

In [ ]:
torch.cuda.empty_cache()
print(" CUDA cache cleared before inference")
print("\n Evaluating Stage 1...")

stage1_model = YOLO(STAGE1_BEST)
stage1_metrics = stage1_model.val(
    data    = YAML_PATH,
    imgsz   = 768,
    device  = [0, 1],
    plots   = True,
    project = f"{WORK_DIR}/runs",
    name    = "stage1_eval",
)

print(f"\n  Stage 1 mAP@0.5      : {stage1_metrics.box.map50:.4f}")
print(f"  Stage 1 mAP@0.5:0.95 : {stage1_metrics.box.map:.4f}")
print(f"  Stage 1 Precision    : {stage1_metrics.box.mp:.4f}")
print(f"  Stage 1 Recall       : {stage1_metrics.box.mr:.4f}")

# Sanity gate — warn if Stage 1 looks too weak before committing to Stage 2
if stage1_metrics.box.map50 < 0.2:
    print("\n  WARNING: mAP@0.5 below 0.2 — consider checking labels or increasing Stage 1 epochs before proceeding")
else:
    print("\n  Stage 1 looks healthy — proceeding to Stage 2")

torch.cuda.empty_cache()

In [ ]:
if Path(STAGE2_LAST).exists():
    print("  Stage 2 checkpoint found — resuming...")
    model_s2 = YOLO(STAGE2_LAST)
    stage2 = model_s2.train(resume=True)
else:
    print(" Stage 2 starting fresh from Stage 1 best weights...")
    model_s2 = YOLO(STAGE1_BEST)
    stage2 = model_s2.train(
        data        = YAML_PATH,
        epochs      = 75,             # more epochs for full convergence
        imgsz       = 768,
        batch       = 16,
        workers     = 2,
        device      = [0,1],
        save_period = 10,
        patience = 30,
    
        # ── No freeze — all layers train ──────────────────────
        freeze      = 0,
    
        # ── Lower LR for fine-tuning ────────────────
        lr0         = 0.001,          # 10x lower than Stage 1
        lrf         = 0.01,
        momentum    = hyp["momentum"],
        weight_decay= hyp["weight_decay"],
    
        # ── Same augmentation ──────────────────────────────────
        hsv_h       = hyp["hsv_h"],
        hsv_s       = hyp["hsv_s"],
        hsv_v       = hyp["hsv_v"],
        degrees     = hyp["degrees"],
        translate   = hyp["translate"],
        scale       = hyp["scale"],
        fliplr      = hyp["fliplr"],
        mosaic      = 1.0,
        mixup       = hyp["mixup"],
    
        # ── Monitoring ─────────────────────────────────
        plots       = True,
        save        = True,
        project     = f"{WORK_DIR}/runs",
        name        = "stage2_finetune",
        exist_ok    = True,
    )

print("\n Stage 2 complete")
stage2_best = f"{WORK_DIR}/runs/stage2_finetune/weights/best.pt"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def plot_training(stage1_dir, stage2_dir):
    s1 = pd.read_csv(f"{stage1_dir}/results.csv")
    s2 = pd.read_csv(f"{stage2_dir}/results.csv")
    
    # Clean column names (Ultralytics adds spaces)
    s1.columns = s1.columns.str.strip()
    s2.columns = s2.columns.str.strip()
    
    # Offset Stage 2 epochs to continue from Stage 1
    s2 = s2.copy()
    s2['epoch'] = s2['epoch'] + len(s1)

    combined = pd.concat([s1, s2], ignore_index=True)
    stage1_end = len(s1)  # vertical line position

    metrics = [
        ("metrics/mAP50(B)",      "mAP@0.5",       "green"),
        ("metrics/mAP50-95(B)",   "mAP@0.5:0.95",  "blue"),
        ("metrics/precision(B)",  "Precision",      "orange"),
        ("metrics/recall(B)",     "Recall",         "red"),
        ("train/box_loss",        "Box Loss",       "purple"),
        ("train/cls_loss",        "Class Loss",     "brown"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Training Progress — Stage 1 + Stage 2", fontsize=16, fontweight='bold')
    axes = axes.flatten()

    for ax, (col, title, color) in zip(axes, metrics):
        if col not in combined.columns:
            ax.set_title(f"{title} (not found)")
            continue
        
        ax.plot(combined['epoch'], combined[col], color=color, linewidth=2)
        
        # Vertical line separating stages
        ax.axvline(x=stage1_end, color='black', linestyle='--', linewidth=1.5, label='Stage 1 → 2')
        
        # Shade stage regions
        ax.axvspan(0, stage1_end, alpha=0.05, color='blue', label='Stage 1')
        ax.axvspan(stage1_end, combined['epoch'].max(), alpha=0.05, color='green', label='Stage 2')
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f" Saved: {WORK_DIR}/training_curves.png")

# Call after both stages complete
plot_training(
    stage1_dir = f"{WORK_DIR}/runs/stage1_frozen",
    stage2_dir = f"{WORK_DIR}/runs/stage2_finetune"
)

In [ ]:
final_model = YOLO(STAGE2_BEST if Path(STAGE2_BEST).exists() else STAGE1_BEST)
print(f" Loaded final model from: {STAGE2_BEST if Path(STAGE2_BEST).exists() else STAGE1_BEST}")

In [ ]:
# ============================================================
# Evaluate & Validate
# ============================================================
print("\n" + "="*55)
print("  TIP 5: Evaluation on Validation Set")
print("="*55)

final_model = YOLO(stage2_best)

metrics = final_model.val(
    data    = YAML_PATH,
    imgsz   = 768,
    device  = [0,1],
    plots   = True,              # saves PR curve, confusion matrix
    save_json = True,
    project = f"{WORK_DIR}/runs",
    name    = "final_eval",
)

print(f"\n  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"  Precision    : {metrics.box.mp:.4f}")
print(f"  Recall       : {metrics.box.mr:.4f}")

In [ ]:
results = final_model.predict(
    source  = TEST_IMGS,
    imgsz   = 768,
    conf    = 0.01,
    iou     = 0.5,
    augment = True,
    device  = [0, 1],
    save    = False,
    stream  = True,     # ← prevents RAM buildup
)

# ── Build pred_map ───────────────────────────────────────────
pred_map = {}
for r in results:
    img_name = Path(r.path).name
    parts = []
    if r.boxes is not None and len(r.boxes):
        for box in r.boxes:
            cls_name = classes[int(box.cls.item())]
            conf     = round(float(box.conf.item()), 4)
            x1, y1, x2, y2 = [round(v, 2) for v in box.xyxy[0].tolist()]
            parts.append(f"{cls_name} {conf} {x1} {y1} {x2} {y2}")
    pred_map[img_name] = " ".join(parts) if parts else ""

# ── Build submission ─────────────────────────────────────────
all_image_ids = sorted([Path(f).name for f in Path(TEST_IMGS).glob("*.jpg")])

rows = []
for img_id in all_image_ids:
    pred = pred_map.get(img_id, "")
    pred = "" if pred is None else str(pred)
    rows.append({"ImageID": img_id, "PredictionString": pred})

submission = pd.DataFrame(rows)
submission["PredictionString"] = (
    submission["PredictionString"]
    .fillna("")
    .astype(str)
    .str.strip()
)

null_count  = submission["PredictionString"].isnull().sum()
empty_count = (submission["PredictionString"] == "").sum()
print(f"Total rows        : {len(submission)}")
print(f"With detections   : {len(submission) - empty_count}")
print(f"Empty (no pests)  : {empty_count}")
print(f"Null values       : {null_count}")
assert null_count == 0, " Still has nulls — do not submit!"
print("\n Submission is clean")

submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)
print(submission.head(3))

In [ ]:
import shutil

shutil.copy(
    f"{WORK_DIR}/runs/stage2_finetune/weights/best.pt",
    f"{WORK_DIR}/best_stage2.pt"
)
shutil.copy(
    f"{WORK_DIR}/runs/stage1_frozen/weights/best.pt",
    f"{WORK_DIR}/best_stage1.pt"
)
print(" Weights copied to working root")

In [ ]:
# ============================================================
#  INFERENCE NOTEBOOK — Pest Detection Submission
# ============================================================
import pandas as pd
import torch
from pathlib import Path
from ultralytics import YOLO

BEST_WEIGHTS = "/kaggle/input/notebooks/kumkumkwatra/21f3000783-week-10/best_stage2.pt"
TEST_IMGS    = "/kaggle/input/competitions/object-detetction-week-10-dlp/test"
WORK_DIR     = "/kaggle/working"

CLASSES = [
    "RiceLeafRoller", "RiceLeafCaterpillar", "PaddyStemMaggot",
    "AsiaticRiceBorer", "YellowRiceBorer", "RiceGallMidge",
    "RiceStemfly", "BrownPlantHopper", "WhiteBackedPlantHopper",
    "SmallBrownPlantHopper", "RiceWaterWeevil", "RiceLeafhopper",
    "GrainSpreaderThrips", "RiceShellPest", "Grub",
    "MoleCricket", "Wireworm", "WhiteMarginedMoth",
    "BlackCutworm", "LargeCutworm", "YellowCutworm",
    "RedSpider", "CornBorer", "Armyworm"
]

# ── Inference ────────────────────────────────────────────────
torch.cuda.empty_cache()
model = YOLO(BEST_WEIGHTS)

results = model.predict(
    source  = TEST_IMGS,
    imgsz   = 1024,
    conf    = 0.01,
    iou     = 0.5,
    augment = True,
    device  = [0, 1],
    save    = False,
)

# ── Build pred_map ───────────────────────────────────────────
pred_map = {}
for r in results:
    img_name = Path(r.path).name
    parts = []
    if r.boxes is not None and len(r.boxes):
        for box in r.boxes:
            cls_name = CLASSES[int(box.cls.item())]
            conf     = round(float(box.conf.item()), 4)
            x1, y1, x2, y2 = [round(v, 2) for v in box.xyxy[0].tolist()]
            parts.append(f"{cls_name} {conf} {x1} {y1} {x2} {y2}")
    # Always assign a string — never None or NaN
    pred_map[img_name] = " ".join(parts) if parts else ""

# ── Build submission from ALL test images ────────────────────
all_image_ids = sorted([Path(f).name for f in Path(TEST_IMGS).glob("*.jpg")])

rows = []
for img_id in all_image_ids:
    pred = pred_map.get(img_id, "")
    pred = "" if pred is None else str(pred)   # triple safety against None
    rows.append({"ImageID": img_id, "PredictionString": pred})

submission = pd.DataFrame(rows)

# ── Null fix ─────────────────────────────────────────────────
submission["PredictionString"] = (
    submission["PredictionString"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# ── Final validation before saving ───────────────────────────
null_count  = submission["PredictionString"].isnull().sum()
empty_count = (submission["PredictionString"] == "").sum()
print(f"Total rows        : {len(submission)}")
print(f"With detections   : {len(submission) - empty_count}")
print(f"Empty (no pests)  : {empty_count}")   # allowed by competition
print(f"Null values       : {null_count}")    # must be 0
assert null_count == 0, " Still has nulls — do not submit!"
print("\n Submission is clean — no nulls")

submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)

# ── Spot check ───────────────────────────────────────────────
print("\nSample rows:")
print(submission[submission["PredictionString"] != ""].head(3).to_string())